# Takeoff-Frame Fix — Validation Notebook

## 0. Clone repo (latest `main`, with the fix) + mount Drive

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_REPO = 'https://github.com/bballhaus/showjumping-ssl.git'
DRIVE_DATA_ROOT = '/content/drive/MyDrive/CS131'
BRANCH = 'main'

from google.colab import drive
drive.mount('/content/drive')
Path(f'{DRIVE_DATA_ROOT}/data').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_DATA_ROOT}/checkpoints').mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/project')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                    GITHUB_REPO, str(REPO_DIR)], check=True)

for sub in ['data', 'checkpoints']:
    target = Path(f'{DRIVE_DATA_ROOT}/{sub}')
    link = REPO_DIR / sub
    if link.is_symlink() or link.exists():
        if link.is_dir() and not link.is_symlink():
            subprocess.run(['rm', '-rf', str(link)], check=True)
        else:
            link.unlink(missing_ok=True)
    link.symlink_to(target, target_is_directory=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'log', '-1', '--format=%h  %cs  %s']).decode().strip()
bar = '=' * 70
print('\n'.join(['', bar, f'RUNNING COMMIT: {commit}', f'cwd: {os.getcwd()}', bar]))

## 1. Install dependencies + verify GPU

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg

import torch
print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
!nvidia-smi 2>&1 | head -10

## 2. Re-run the fixed pipeline on all clips

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from src.preprocess.run_pipeline import process_clip, load_fence_annotations
from src.preprocess.detect import HorseDetector

detector = HorseDetector(weights='yolov8n.pt', device='cuda')
fences = load_fence_annotations(Path('data/annotations/fences.csv'))
clips = sorted(Path('data/clips').glob('*.mp4'))
print(f'[pipeline] {len(clips)} clips, {len(fences)} fence annotations')

rows = []
for cp in tqdm(clips, desc='YOLO + geometry', unit='clip'):
    rows.append(process_clip(cp, detector, fences.get(cp.stem)))

df = pd.DataFrame(rows)
out = Path('data/annotations/auto.csv')
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f'wrote {len(df)} rows -> {out}')
print(df[df['d_meters'].notna()][['clip_id', 'type', 'd_meters', 'takeoff_frame']].head(10).to_string(index=False))

## 2b. Cache box trajectories for local iteration

In [ ]:
from pathlib import Path
from src.preprocess.cache_tracks import dump_tracks

data = dump_tracks(Path('data/clips'), detector, Path('data/annotations/tracks.json'), stride=2)
n_boxes = sum(len(c['boxes']) for c in data['clips'].values())
print(f"cached {len(data['clips'])} clips, {n_boxes} boxes -> data/annotations/tracks.json")

## 3. Regression guard — takeoff frames should no longer pin to the clip end

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/annotations/auto.csv')
sub = df[(df['takeoff_frame'] >= 0) & df['d_meters'].notna()].copy()

def last_sampled_idx(clip_id, stride=2):
    cap = cv2.VideoCapture(f'data/clips/{clip_id}.mp4')
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    cap.release()
    return max(1, ((n - 1) // stride) * stride)

sub['last_idx'] = sub['clip_id'].map(last_sampled_idx)
sub['frac'] = sub['takeoff_frame'] / sub['last_idx']

share_end = float((sub['frac'] >= 0.85).mean())
print(f'clips checked: {len(sub)}')
print(f"takeoff position (fraction into clip) - mean {sub['frac'].mean():.2f}, median {sub['frac'].median():.2f}")
print(f'share with takeoff in last 15% of clip: {share_end:.0%}   (old buggy run: 100%)')
print('PASS - takeoff frames are spread through the clip'
      if share_end < 0.6 else
      'WARN - still clustering at clip end; inspect the overlays below')

plt.figure(figsize=(6, 3))
plt.hist(sub['frac'].clip(0, 1.2), bins=20, range=(0, 1.2), color='#4C72B0', edgecolor='white')
plt.axvline(0.85, color='red', ls='--', label='last 15%')
plt.xlabel('takeoff position (fraction into clip)')
plt.ylabel('clips')
plt.title('Detected takeoff position after fix')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Visual check — overlay the detected takeoff frame

In [ ]:
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from src.preprocess.detect import Box, HorseDetector
from src.viz.detection_examples import render_clip

N = 6
detector = HorseDetector(weights='yolov8n.pt', device='cuda')
fences = pd.read_csv('data/annotations/fences.csv')
out_dir = Path('data/results/takeoff_check')
out_dir.mkdir(parents=True, exist_ok=True)

rows = int((N + 1) // 2)
fig, axes = plt.subplots(rows, 2, figsize=(12, 4 * rows))
axes = axes.ravel()
shown = 0
for _, row in fences.head(N).iterrows():
    clip = Path(f"data/clips/{row['clip_id']}.mp4")
    if not clip.exists():
        continue
    fence_box = Box(float(row['x1']), float(row['y1']),
                    float(row['x2']), float(row['y2']), label='fence')
    pole = int(row['pole_count']) if pd.notna(row.get('pole_count')) else None
    out = out_dir / f"{row['clip_id']}.jpg"
    render_clip(clip, fence_box, pole, detector, out)
    if out.exists():
        img = cv2.cvtColor(cv2.imread(str(out)), cv2.COLOR_BGR2RGB)
        axes[shown].imshow(img)
        axes[shown].set_title(row['clip_id'], fontsize=9)
        axes[shown].axis('off')
        shown += 1
for j in range(shown, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()
print(f'rendered {shown} clips -> {out_dir}')

## 5. Mark ground-truth takeoff frames (optional but high-value)

In [ ]:
!pip install -q jupyter_bbox_widget ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

import pandas as pd
from src.preprocess.annotate_colab import TakeoffAnnotator

fence_clips = pd.read_csv('data/annotations/fences.csv')['clip_id'].astype(str).tolist()
TakeoffAnnotator('data/clips', only_clips=fence_clips).start()

## 6. Gut-check: detected vs hand-marked takeoff (new steepest-rise detector)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from src.viz.detection_examples import render_truth_comparison

out = Path('data/results/takeoff_check/truth_vs_detected.png')
render_truth_comparison(
    clips_dir=Path('data/clips'),
    tracks_path=Path('data/annotations/tracks.json'),
    fences_path=Path('data/annotations/fences.csv'),
    takeoffs_path=Path('data/annotations/takeoffs.csv'),
    out_path=out,
    max_clips=33,
)
display(Image(filename=str(out)))

## 7. Jump-detection tuning — cache full-video scans

In [ ]:
from pathlib import Path
from src.data.cache_scan import dump_scans

data = dump_scans(Path('data/raw'), detector, Path('data/annotations/scan_cache.json'),
                  sample_fps=12.0, limit_videos=1, max_seconds=90.0)
n = sum(len(v['samples']) for v in data['videos'].values())
print(f"cached {len(data['videos'])} videos, {n} samples -> data/annotations/scan_cache.json")
for vid, e in data['videos'].items():
    print(f"  {vid}: {len(e['samples'])} samples, {e['samples'][-1][0]:.1f}s")